This notebook provides a workflow for model inferrence on the standartised remote sensing data for the specified year,

In [ ]:
import os
import rasterio
import xgboost as xgb

In [ ]:
# define paths
model_path = "./cdse_model_xgb_nosmote.json"

input_folder = "../data/grasslvnd/combined_2024"
output_folder = "../data/grasslvnd/preds_2024"

target_crs = "EPSG:3059"

In [ ]:
# load the trained model and get used feature names
model = xgb.XGBClassifier()
model.load_model(model_path)

used_bands = model.get_booster().feature_names

In [ ]:
# process each raster in the input folder
rasters = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.tif')]
for input_raster in rasters:
    basename = os.path.basename(input_raster)
    outpath = os.path.join(output_folder, f"{os.path.splitext(basename)[0]}_pred.tif")
    
    with rasterio.open(input_raster) as src:
        desc_to_index = {desc: i + 1 for i, desc in enumerate(src.descriptions)}
        selected_indices = []
        for b in used_bands:
            matches = [i for desc, i in desc_to_index.items() if desc and b in desc]
            if matches:
                selected_indices.append(matches[0])
            else:
                raise ValueError(f"'{b}' not found.")
        selected_data = src.read(indexes=selected_indices)
        profile = src.profile

    n_bands, n_rows, n_cols = selected_data.shape
    X = selected_data.reshape(n_bands, -1).T 
    y_pred = model.predict(X)
    y_pred_raster = y_pred.reshape(n_rows, n_cols)

    profile.update(
        dtype=rasterio.uint8,
        count=1,
        compress='lzw',
        crs=target_crs
    )

    with rasterio.open(outpath, 'w', **profile) as dst:
        dst.write(y_pred_raster.astype(rasterio.uint8), 1)

    print(f"saved to: {outpath}.")

